In [19]:
# split datasets
import random

# 读取数据
words = open('names.txt', 'r').read().splitlines()

# 固定随机种子，保证每次划分结果一致
random.seed(42)
random.shuffle(words)

# 按照 80%、10%、10% 划分
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

words_train = words[:n1]
words_val = words[n1:n2]
words_test = words[n2:]

print(len(words_train), len(words_val), len(words_test))
words_train[:10], words_test[:5], words_val[:5]

25626 3203 3204


(['yuheng',
  'diondre',
  'xavien',
  'jori',
  'juanluis',
  'erandi',
  'phia',
  'samatha',
  'phoenix',
  'emmelynn'],
 ['mustafa', 'reuben', 'kahlel', 'balian', 'izael'],
 ['amay', 'aytana', 'jenevi', 'zylen', 'naim'])

## biagram sample

In [20]:
import torch
import torch.nn.functional as F

chars = ['.'] + sorted(set(''.join(words)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}


def build_dataset(words):
    xs, ys = [], []

    for w in words:
        chs = ['.'] + list(w) + ['.']

        for ch1, ch2 in zip(chs, chs[1:]):
            xs.append(stoi[ch1])
            ys.append(stoi[ch2])

    return torch.tensor(xs), torch.tensor(ys)


xtr, ytr = build_dataset(words_train)
xval, yval = build_dataset(words_val)
xtest, ytest = build_dataset(words_test)

In [21]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

for i in range(200):
    # forward
    xenc = F.one_hot(xtr, num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)

    loss = -probs[torch.arange(len(ytr)), ytr].log().mean()
    loss += 0.01 * (W**2).mean()

    # backward
    W.grad = None
    loss.backward()

    # update
    W.data += -50 * W.grad

print('Train loss:', loss.item())

Train loss: 2.48301100730896


In [22]:
def evaluate(xs, ys):
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)

    loss = -probs[torch.arange(len(ys)), ys].log().mean()
    return loss.item()


print('Train loss:', evaluate(xtr, ytr))
print('Val loss:  ', evaluate(xval, yval))
print('Test loss: ', evaluate(xtest, ytest))

Train loss: 2.4653258323669434
Val loss:   2.4634485244750977
Test loss:  2.469421625137329


In [23]:
def evaluate_bigram(words, P):
    log_likelihood = 0.0
    n = 0

    for w in words:
        chs = ['.'] + list(w) + ['.']

        for ch1, ch2 in zip(chs, chs[1:]):
            ix1 = stoi[ch1]
            ix2 = stoi[ch2]

            log_likelihood += torch.log(P[ix1, ix2])
            n += 1

    return (-log_likelihood / n).item()


print('Bigram model')
print('Train loss:', evaluate_bigram(words_train, probs))
print('Val loss:  ', evaluate_bigram(words_val, probs))
print('Test loss: ', evaluate_bigram(words_test, probs))

Bigram model
Train loss: 3.0110957622528076
Val loss:   3.0046846866607666
Test loss:  3.0114006996154785


In [24]:
g = torch.Generator().manual_seed(2147483647)

for i in range(10):
    out = []
    ix = 0

    while True:
        logits = W[ix]
        counts = logits.exp()
        p = counts / counts.sum()

        ix = torch.multinomial(
            p, num_samples=1, replacement=True, generator=g
        ).item()

        out.append(itos[ix])

        if ix == 0:
            break

    print(''.join(out))

junide.
janasah.
prelay.
a.
nn.
kohin.
tolian.
juee.
ksahnaauranilevias.
dedainrwieta.


## Trigram sample

In [25]:
xs, ys = [], []

for w in words:
    chs = ['.'] + ['.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1] * 27 + stoi[ch2]
        ix2 = stoi[ch3]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples: ', num)

# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27*27, 27), generator=g, requires_grad=True)

number of examples:  228146


In [26]:
import torch.nn.functional as F

# gradient descent
for i in range(200):
    # forward pass
    xenc = F.one_hot(xs, num_classes=27*27).float()
    logits = xenc @ W

    # softmax
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)
    
    # loss
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()

    # print(loss.item())
    
    # backward
    W.grad = None
    loss.backward()

    # update
    W.data += -50 * W.grad

loss.item()

2.3923046588897705

In [27]:
def evaluate_trigram(words, P):
    log_likelihood = 0.0
    n = 0

    for w in words:
        chs = ['.', '.'] + list(w) + ['.']

        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            ix1 = stoi[ch1] * 27 + stoi[ch2]
            ix2 = stoi[ch3]

            log_likelihood += torch.log(P[ix1, ix2])
            n += 1

    return (-log_likelihood / n).item()


print('\nTrigram model')
print('Train loss:', evaluate_trigram(words_train, probs))
print('Val loss:  ', evaluate_trigram(words_val, probs))
print('Test loss: ', evaluate_trigram(words_test, probs))


Trigram model
Train loss: 3.43628191947937
Val loss:   3.4227678775787354
Test loss:  3.4497292041778564


In [28]:
g = torch.Generator().manual_seed(2147483647)

for i in range(10):
    out = []
    ix1 = 0

    while True:
        logits = W[ix1]
        counts = logits.exp()
        p = counts / counts.sum()

        ix2 = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix2])

        if ix2 == 0:
            break

        ix1 = (ix1 % 27) * 27 + ix2

    print(''.join(out))

juwjde.
jakasiz.
prefarie.
na.
kairritonian.
juwzvksiania.
yaniahviassdbyainrwibwlassjywnfyla.
te.
faveumtrifoetumj.
poytslena.


## Result